In [ ]:
import pandas as pd
import json

def getAllPassesInMatch(matchEventDf: pd.DataFrame):
    df = matchEventDf.copy()
    df.drop(['ball_receipt', 'ball_recovery', 'carry', 'counterpress', 'dribble', 'duel', 'goalkeeper', 'interception', 'substitution', 'type'], axis=1, inplace=True)
    try:
        df.drop(['bad_behaviour', 'block', 'injury_stoppage', 'foul_committed', 'foul_won', 'tactics'])
    except:
        pass
    return df.dropna(subset=['pass'])

In [ ]:
import os

# create one df containing all passes

list_of_passes = []
eventFiles = os.listdir("data/events")

for i in range (100):
    with open(f'data/events/{eventFiles[i]}', 'r') as match:
        match1 = []
        events = json.load(match)
        for item in events:
            match1.append(item)

    list_of_passes.extend(match1)

# concatenate all passes into a single dataframe
get_all_passes_df = getAllPassesInMatch(pd.DataFrame(list_of_passes))

# Cleaning Pass Dataframe
- Dropping unnecessary columns
- Moving pass column into main df
- Splitting location into columns (x, y)
- Filling empty booleans
- Normalisation of enumerations (not implemented)

In [ ]:
# cleaning initial pass df 
all_passes_df_dirty = get_all_passes_df.drop(
    ['tactics', 'id', 'index', 'minute', 'second', 'foul_committed', 'foul_won', 'shot', 'possession',
     'team', 'off_camera', 'miscontrol', 'clearance', 'bad_behaviour', 'block', '50_50', 'injury_stoppage', 'player_off'],
    axis=1
)

# split pass dict into separate columns and rename columns
pass_dict_df = all_passes_df_dirty['pass'].apply(pd.Series)
pass_dict_df.rename(columns=lambda x: 'pass_' + x, inplace=True)

all_passes_df_dirty = all_passes_df_dirty.join(pass_dict_df)
all_passes_df_dirty.drop(['pass'], axis=1, inplace=True)

# split location into x y
all_passes_df_dirty[['location_x', 'location_y']] = all_passes_df_dirty['location'].apply(pd.Series)
all_passes_df_dirty.drop(['location'], axis=1, inplace=True)

# split pass end location into x y
all_passes_df_dirty[['pass_end_location_x', 'pass_end_location_y']] = all_passes_df_dirty['pass_end_location'].apply(pd.Series)
all_passes_df_dirty.drop(['pass_end_location'], axis=1, inplace=True)

# clean boolean values to 0/1
all_passes_df_dirty.fillna(
    {
        'under_pressure': 0,
        'out': 0,
        'pass_switch': 0,
        'pass_aerial_won': 0,
        'pass_through_ball': 0,
        'pass_cross': 0,
        'pass_inswinging': 0,
        'pass_shot_assist': 0,
        'pass_goal_assist': 0,
        'pass_cut_back': 0,
        'pass_straight': 0,
        'pass_backheel': 0,
        'pass_miscommunication': 0,
        'pass_no_touch': 0,
        'pass_deflected': 0,
        'pass_outswinging': 0,
        'pass_body_part': 0,
    },
    inplace=True
)
all_passes_df_dirty.replace({True: 1, False: 0}, inplace=True)
# normalise enum columns - get dict id and set as col value
all_passes_df_dirty['play_pattern'] = all_passes_df_dirty['play_pattern'].apply(lambda x: x['id'] if isinstance(x, dict) else 0)
all_passes_df_dirty['position'] = all_passes_df_dirty['position'].apply(lambda x: x['id'] if isinstance(x, dict) else 0)
all_passes_df_dirty['pass_height'] = all_passes_df_dirty['pass_height'].apply(lambda x: x['id'] if isinstance(x, dict) else 0)
all_passes_df_dirty['pass_type'] = all_passes_df_dirty['pass_type'].apply(lambda x: (x['id'] - 60) if isinstance(x, dict) else 0)
all_passes_df_dirty['pass_outcome'] = all_passes_df_dirty['pass_outcome'].apply(lambda x: x['id'] if isinstance(x, dict) else 0)
all_passes_df_dirty['pass_technique'] = all_passes_df_dirty['pass_technique'].apply(lambda x: (x['id'] - 103) if isinstance(x, dict) else 0)
all_passes_df_dirty['pass_body_part'] = all_passes_df_dirty['pass_body_part'].apply(lambda x: x['id'] if isinstance(x, dict) else -1)


# independent analysis - comment out if team analysis involved
all_passes_df_dirty.drop(['player', 'related_events', 'possession_team', 'pass_recipient', 'pass_assisted_shot_id'], axis=1, inplace=True)
# turn string timestamp into seconds,i.e. 0:00:00.484 = 0.484
from datetime import datetime

def timestamp_to_seconds(ts):
	dt = datetime.strptime(ts, '%H:%M:%S.%f')
	return dt.hour * 3600 + dt.minute * 60 + dt.second + dt.microsecond / 1000000

all_passes_df_dirty['timestamp'] = all_passes_df_dirty['timestamp'].apply(timestamp_to_seconds)

def normalise_body_part(x):
    if x == 68:
        return 1  # Drop Kick
    elif x == 37:
        return 2  # Head
    elif x == 69:
        return 3  # Keeper Arm
    elif x == 38:
        return 4  # Left Foot
    elif x == 40:
        return 5  # Right Foot
    elif x == 70:
        return 6  # Other
    elif x == 106:
        return 0  # No Touch
    else:
        return 0  # Unknown/Other

# normalise pass_body_part to linear
all_passes_df_dirty['pass_body_part'] = all_passes_df_dirty['pass_body_part'].apply(lambda x: normalise_body_part(x))
# normalise enum pass_outcome {9: "Incomplete", 74: "Injury Clearance", 75: "Out", 76: "Pass Offside", 77: "Unknown"} - Added if pass not completed
all_passes_df_dirty['pass_outcome'] = all_passes_df_dirty['pass_outcome'].apply(lambda x: 1 if x == 9 else 2 if x == 74 else 3 if x == 75 else 4 if x == 76 else 5 if x == 77 else 0)

In [ ]:
all_passes_df_dirty